# MoveScope Template Sensitivity

Evaluates how the number of expert feature sequences used to build a template affects score stability.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "movescope").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from movescope.experiments import load_feature_sequences, run_template_sensitivity_from_features

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
expert_dir = ROOT / "data" / "features" / "expert_squat"
test_dir = ROOT / "data" / "features" / "test_squat"

expert_sequences = load_feature_sequences(expert_dir)
test_sequences = load_feature_sequences(test_dir)

print(f"expert sequences: {len(expert_sequences)} from {expert_dir}")
print(f"test sequences: {len(test_sequences)} from {test_dir}")


In [ ]:
rows = run_template_sensitivity_from_features(
    expert_sequences,
    test_sequences,
    counts=(1, 3, 5, 10),
)

df = pd.DataFrame(rows)
if df.empty:
    print("No sensitivity rows. Add .npy/.npz feature arrays to data/features/expert_squat and data/features/test_squat.")
else:
    display(df)


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["template_count"], df["std_score"], marker="o", color="#b94331", linewidth=2)
    ax.set_title("Template count vs score stability")
    ax.set_xlabel("number of expert feature sequences")
    ax.set_ylabel("score standard deviation")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    output = FIG_DIR / "template_sensitivity.png"
    fig.savefig(output, dpi=160)
    print(f"Saved {output}")


In [ ]:
if not df.empty:
    best = df.sort_values("std_score").iloc[0]
    print(
        f"Lowest score std is {best.std_score:.2f} with {int(best.template_count)} expert sequences "
        f"across {int(best.n_tests)} test sequences."
    )
